# 07 — Fusion, Robustness and Reliability Analysis

## Objective

This notebook evaluates the integration of the behavioral and textual bot-detection models on an unseen bot category.

The experiments include:

1. Behavior-only evaluation
2. Text-only evaluation
3. Weighted probability fusion
4. Adaptive fusion
5. Gated fusion
6. Confidence-aware fusion
7. Conservative text override
8. Fusion error analysis
9. Distribution drift analysis
10. Confidence-based reliability analysis
11. Review-case analysis

The unseen Fake Followers category is kept completely separate from model
selection and is used only for final evaluation.

No test-set tuning is performed in this notebook.

In [ ]:
## Experimental Protocol

The evaluation uses the common account-level dataset containing 10,197
accounts.

The unseen-category experiment follows a Leave-One-Bot-Category-Out (LOBO)
design:

- Training: known bot categories + genuine accounts
- Unseen test category: Fake Followers
- Test set: Fake Followers + genuine accounts

The final fusion weights and reliability thresholds were selected using
validation data only and then frozen for unseen-category evaluation.

In [1]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
# ================================
# 2. PROJECT PATHS
# ================================

project_path = os.path.abspath("..")

data_path = os.path.join(project_path, "data")
final_results_path = os.path.join(data_path, "final_results")

print("Project path:", project_path)
print("Final results path:", final_results_path)

Project path: c:\Users\namit\OneDrive\Desktop\AI_Bot_Detection
Final results path: c:\Users\namit\OneDrive\Desktop\AI_Bot_Detection\data\final_results


In [4]:
# ================================
# 3. LOAD BEHAVIORAL FINAL RESULTS
# ================================

behavior_results = pd.read_csv(
    os.path.join(
        final_results_path,
        "behavioral_final_results.csv"
    )
)

print("Shape:", behavior_results.shape)
print("\nColumns:")
print(behavior_results.columns.tolist())

display(behavior_results.head())

Shape: (4285, 4)

Columns:
['actual_label', 'behavior_probability', 'behavior_prediction', 'drift_score']


,actual_label,behavior_probability,behavior_prediction,drift_score
0,0,0.000532,0,0.998392
1,0,0.009074,0,0.869371
2,0,0.007927,0,0.971587
3,0,0.007451,0,0.827019
4,0,0.220586,0,0.933881


In [5]:
# ================================
# 4. BEHAVIORAL BASELINE
# ================================

y_true = behavior_results["actual_label"].values
behavior_prob = behavior_results["behavior_probability"].values
behavior_pred = behavior_results["behavior_prediction"].values

behavior_accuracy = accuracy_score(y_true, behavior_pred)
behavior_precision = precision_score(y_true, behavior_pred)
behavior_recall = recall_score(y_true, behavior_pred)
behavior_f1 = f1_score(y_true, behavior_pred)

print("=== BEHAVIORAL BASELINE ===")
print(f"Accuracy : {behavior_accuracy:.4%}")
print(f"Precision: {behavior_precision:.4%}")
print(f"Recall   : {behavior_recall:.4%}")
print(f"F1-score : {behavior_f1:.4%}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, behavior_pred))

=== BEHAVIORAL BASELINE ===
Accuracy : 98.4364%
Precision: 99.7461%
Recall   : 98.1574%
F1-score : 98.9454%

Confusion Matrix:
[[1075    8]
 [  59 3143]]


In [6]:
# ================================
# 5. LOAD TEXT PREDICTIONS
# ================================

text_pred = pd.read_csv(
    os.path.join(
        data_path,
        "text_model_predictions.csv"
    )
)

print("Shape:", text_pred.shape)
print("\nColumns:")
print(text_pred.columns.tolist())

display(text_pred.head())

Shape: (10197, 5)

Columns:
['user_id', 'label', 'account_type', 'text_model_probability', 'text_model_prediction']


,user_id,label,account_type,text_model_probability,text_model_prediction
0,678033.0,0,genuine_accounts.csv,0.031738,0
1,722623.0,0,genuine_accounts.csv,0.022437,0
2,755116.0,0,genuine_accounts.csv,0.053431,0
3,755746.0,0,genuine_accounts.csv,0.038380,0
4,785080.0,0,genuine_accounts.csv,0.032506,0


In [7]:
# ================================
# 6. INSPECT TEXT PREDICTION COVERAGE
# ================================

print("Account types:")
print(text_pred["account_type"].value_counts())

print("\nLabels:")
print(text_pred["label"].value_counts())

print("\nUnique users:", text_pred["user_id"].nunique())

print("\nMissing values:")
print(text_pred.isna().sum())


Account types:
account_type
social_spambots_2.csv         3457
fake_followers.csv            3202
genuine_accounts.csv          1083
traditional_spambots_1.csv    1000
social_spambots_1.csv          991
social_spambots_3.csv          464
Name: count, dtype: int64

Labels:
label
1    9114
0    1083
Name: count, dtype: int64

Unique users: 10197

Missing values:
user_id                   0
label                     0
account_type              0
text_model_probability    0
text_model_prediction     0
dtype: int64


In [8]:
# Check how many of the behavioral test accounts are present
test_ids = set(behavior_results.index)

print("Behavior test rows:", len(behavior_results))
print("Text prediction rows:", len(text_pred))

Behavior test rows: 4285
Text prediction rows: 10197


In [9]:
# ================================
# 6. STANDARD TEST FUSION SETUP
# ================================

# Prepare text predictions for merging
text_pred_merge = text_pred.copy()

text_pred_merge["user_id"] = pd.to_numeric(
    text_pred_merge["user_id"],
    errors="coerce"
).astype("Int64")

text_pred_merge = text_pred_merge.rename(
    columns={"user_id": "id"}
)

print("Text prediction rows:", len(text_pred_merge))
print("Unique text IDs:", text_pred_merge["id"].nunique())

Text prediction rows: 10197
Unique text IDs: 10197


In [11]:
# ================================
# 7. LOAD COMMON TEST RESULTS
# ================================

behavior_common_test = pd.read_csv(
    os.path.join(
        final_results_path,
        "behavioral_common_test_results.csv"
    )
)

print("Shape:", behavior_common_test.shape)
print("Columns:", behavior_common_test.columns.tolist())

display(behavior_common_test.head())

Shape: (1530, 5)
Columns: ['id', 'label', 'account_type', 'behavior_probability', 'behavior_prediction']


,id,label,account_type,behavior_probability,behavior_prediction
0,2157382005,0,genuine_accounts,0.130344,0
1,177906959,0,genuine_accounts,0.189599,0
2,2248762537,0,genuine_accounts,0.019757,0
3,1727495887,0,genuine_accounts,0.077747,0
4,2152052819,0,genuine_accounts,0.107989,0


In [12]:
# ================================
# 8. CREATE STANDARD FUSION TEST SET
# ================================

fusion_test_standard = behavior_common_test.merge(
    text_pred_merge[
        [
            "id",
            "text_model_probability",
            "text_model_prediction"
        ]
    ],
    on="id",
    how="inner"
)

print("Fusion test shape:", fusion_test_standard.shape)

print("\nMissing values:")
print(
    fusion_test_standard[
        [
            "behavior_probability",
            "text_model_probability"
        ]
    ].isna().sum()
)

display(fusion_test_standard.head())

Fusion test shape: (1530, 7)

Missing values:
behavior_probability      0
text_model_probability    0
dtype: int64


,id,label,account_type,behavior_probability,behavior_prediction,text_model_probability,text_model_prediction
0,2157382005,0,genuine_accounts,0.130344,0,0.033657,0
1,177906959,0,genuine_accounts,0.189599,0,0.075028,0
2,2248762537,0,genuine_accounts,0.019757,0,0.045131,0
3,1727495887,0,genuine_accounts,0.077747,0,0.084882,0
4,2152052819,0,genuine_accounts,0.107989,0,0.100697,0


In [13]:
# ================================
# 9. FINAL STANDARD 50:50 FUSION
# ================================

best_alpha = 0.5

y_standard = fusion_test_standard["label"].values

behavior_prob_standard = (
    fusion_test_standard["behavior_probability"].values
)

text_prob_standard = (
    fusion_test_standard["text_model_probability"].values
)

# 50% behavioral + 50% textual
final_standard_prob = (
    best_alpha * behavior_prob_standard
    + (1 - best_alpha) * text_prob_standard
)

final_standard_pred = (
    final_standard_prob >= 0.5
).astype(int)

# Store predictions
fusion_test_standard["final_probability"] = final_standard_prob
fusion_test_standard["final_prediction"] = final_standard_pred

# Evaluate
standard_accuracy = accuracy_score(
    y_standard,
    final_standard_pred
)

standard_precision = precision_score(
    y_standard,
    final_standard_pred
)

standard_recall = recall_score(
    y_standard,
    final_standard_pred
)

standard_f1 = f1_score(
    y_standard,
    final_standard_pred
)

print("=== STANDARD 50:50 FUSION ===")
print("Behavior weight:", best_alpha)
print("Text weight:", 1 - best_alpha)
print(f"Accuracy : {standard_accuracy:.4%}")
print(f"Precision: {standard_precision:.4%}")
print(f"Recall   : {standard_recall:.4%}")
print(f"F1-score : {standard_f1:.4%}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_standard, final_standard_pred))

=== STANDARD 50:50 FUSION ===
Behavior weight: 0.5
Text weight: 0.5
Accuracy : 99.4771%
Precision: 99.7802%
Recall   : 99.6342%
F1-score : 99.7072%

Confusion Matrix:
[[ 160    3]
 [   5 1362]]


In [14]:
# ================================
# 10. CHECK FOR CLEAN LOBO TEXT RESULTS
# ================================

print("Files in final_results:")

for filename in os.listdir(final_results_path):
    print(filename)

Files in final_results:
behavioral_common_test_results.csv
behavioral_final_results.csv


In [15]:
# ================================
# 11. LOAD CLEAN LOBO TEXT RESULTS
# ================================

text_lobo_results = pd.read_csv(
    os.path.join(
        final_results_path,
        "text_lobo_test_results.csv"
    )
)

print("Shape:", text_lobo_results.shape)
print("Columns:", text_lobo_results.columns.tolist())

display(text_lobo_results.head())

Shape: (4285, 2)
Columns: ['text_probability', 'text_prediction']


,text_probability,text_prediction
0,0.038584,0
1,0.023926,0
2,0.076167,0
3,0.009980,0
4,0.206501,0


In [16]:
# ================================
# 12. LOBO TEXT BASELINE
# ================================

text_lobo_prob = text_lobo_results["text_probability"].values
text_lobo_pred = text_lobo_results["text_prediction"].values

# Use the authoritative behavioral test labels
y_lobo = behavior_results["actual_label"].values

text_lobo_accuracy = accuracy_score(
    y_lobo,
    text_lobo_pred
)

text_lobo_precision = precision_score(
    y_lobo,
    text_lobo_pred
)

text_lobo_recall = recall_score(
    y_lobo,
    text_lobo_pred
)

text_lobo_f1 = f1_score(
    y_lobo,
    text_lobo_pred
)

print("=== LOBO TEXT-ONLY BASELINE ===")
print(f"Accuracy : {text_lobo_accuracy:.4%}")
print(f"Precision: {text_lobo_precision:.4%}")
print(f"Recall   : {text_lobo_recall:.4%}")
print(f"F1-score : {text_lobo_f1:.4%}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_lobo, text_lobo_pred))

=== LOBO TEXT-ONLY BASELINE ===
Accuracy : 75.0058%
Precision: 88.4796%
Recall   : 76.5147%
F1-score : 82.0633%

Confusion Matrix:
[[ 764  319]
 [ 752 2450]]


In [17]:
# ================================
# 13. LOBO 50:50 FUSION
# ================================

best_alpha_lobo = 0.5

behavior_lobo_prob = behavior_results[
    "behavior_probability"
].values

text_lobo_prob = text_lobo_results[
    "text_probability"
].values

# 50% behavioral + 50% textual
lobo_fusion_prob = (
    best_alpha_lobo * behavior_lobo_prob
    + (1 - best_alpha_lobo) * text_lobo_prob
)

lobo_fusion_pred = (
    lobo_fusion_prob >= 0.5
).astype(int)

# Evaluate
lobo_fusion_accuracy = accuracy_score(
    y_lobo,
    lobo_fusion_pred
)

lobo_fusion_precision = precision_score(
    y_lobo,
    lobo_fusion_pred
)

lobo_fusion_recall = recall_score(
    y_lobo,
    lobo_fusion_pred
)

lobo_fusion_f1 = f1_score(
    y_lobo,
    lobo_fusion_pred
)

print("=== LOBO 50:50 FUSION ===")
print("Behavior weight:", best_alpha_lobo)
print("Text weight:", 1 - best_alpha_lobo)

print(f"Accuracy : {lobo_fusion_accuracy:.4%}")
print(f"Precision: {lobo_fusion_precision:.4%}")
print(f"Recall   : {lobo_fusion_recall:.4%}")
print(f"F1-score : {lobo_fusion_f1:.4%}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_lobo, lobo_fusion_pred))

=== LOBO 50:50 FUSION ===
Behavior weight: 0.5
Text weight: 0.5
Accuracy : 97.0128%
Precision: 99.1054%
Recall   : 96.8770%
F1-score : 97.9785%

Confusion Matrix:
[[1055   28]
 [ 100 3102]]


In [18]:
# ================================
# 15. LOAD LOBO RESULTS WITH IDS
# ================================

behavior_lobo = pd.read_csv(
    os.path.join(
        final_results_path,
        "behavioral_lobo_test_results.csv"
    )
)

text_lobo = pd.read_csv(
    os.path.join(
        final_results_path,
        "text_lobo_test_results_with_ids.csv"
    )
)

print("Behavior LOBO:", behavior_lobo.shape)
print("Text LOBO:", text_lobo.shape)

Behavior LOBO: (4285, 5)
Text LOBO: (4285, 5)


In [27]:
# ==========================================
# SAVE LOBO BEHAVIOR VALIDATION RESULTS
# ==========================================

behavior_lobo_val_results = pd.DataFrame({
    "label": y_lobo_val,
    "behavior_probability": behavior_lobo_val_prob,
    "behavior_prediction": (
        behavior_lobo_val_prob >= 0.5
    ).astype(int)
})

behavior_lobo_val_results.to_csv(
    "../data/final_results/behavioral_lobo_val_results.csv",
    index=False
)

print("Saved:", behavior_lobo_val_results.shape)
display(behavior_lobo_val_results.head())

NameError: name 'y_lobo_val' is not defined

In [20]:
# ================================
# 16. ALIGN BEHAVIOR + TEXT BY ID
# ================================

behavior_lobo["id"] = pd.to_numeric(
    behavior_lobo["id"],
    errors="coerce"
).astype("Int64")

text_lobo["id"] = pd.to_numeric(
    text_lobo["id"],
    errors="coerce"
).astype("Int64")

text_lobo = text_lobo.rename(
    columns={"id": "id"}
)

lobo_fusion = behavior_lobo.merge(
    text_lobo[
        [
            "id",
            "text_probability",
            "text_prediction"
        ]
    ],
    on="id",
    how="inner"
)

print("Merged LOBO accounts:", len(lobo_fusion))
print("Unique IDs:", lobo_fusion["id"].nunique())

display(lobo_fusion.head())

Merged LOBO accounts: 4285
Unique IDs: 4285


,id,label,account_type,behavior_probability,behavior_prediction,text_probability,text_prediction
0,191839658,0,genuine_accounts,0.000532,0,0.038584,0
1,2157382005,0,genuine_accounts,0.009074,0,0.023926,0
2,1947320929,0,genuine_accounts,0.007927,0,0.076167,0
3,1733095801,0,genuine_accounts,0.007451,0,0.009980,0
4,21959183,0,genuine_accounts,0.220586,0,0.206501,0


In [21]:
# ==========================================
# 15. PROPER ALIGNED LOBO FUSION
# ==========================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Ground-truth labels
y_lobo = lobo_fusion["label"].values

# Behavior probability
behavior_prob = lobo_fusion["behavior_probability"].values

# Text probability
text_prob = lobo_fusion["text_probability"].values

# 50:50 fusion
lobo_fusion_prob = (
    0.50 * behavior_prob +
    0.50 * text_prob
)

lobo_fusion_pred = (
    lobo_fusion_prob >= 0.5
).astype(int)

print("==========================================")
print("PROPER ALIGNED LOBO FUSION — 50:50")
print("==========================================")

print("Accuracy :", accuracy_score(y_lobo, lobo_fusion_pred))
print("Precision:", precision_score(y_lobo, lobo_fusion_pred))
print("Recall   :", recall_score(y_lobo, lobo_fusion_pred))
print("F1 Score :", f1_score(y_lobo, lobo_fusion_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_lobo, lobo_fusion_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_lobo,
        lobo_fusion_pred,
        target_names=["Genuine", "Bot"]
    )
)

PROPER ALIGNED LOBO FUSION — 50:50
Accuracy : 0.9701283547257876
Precision: 0.9910543130990416
Recall   : 0.9687695190505934
F1 Score : 0.9797852179406191

Confusion Matrix:
[[1055   28]
 [ 100 3102]]

Classification Report:
              precision    recall  f1-score   support

     Genuine       0.91      0.97      0.94      1083
         Bot       0.99      0.97      0.98      3202

    accuracy                           0.97      4285
   macro avg       0.95      0.97      0.96      4285
weighted avg       0.97      0.97      0.97      4285



In [25]:
# ==========================================
# FIND AVAILABLE FUSION RESULT FILES
# ==========================================

import os

print("FINAL RESULTS FILES:\n")

for file in sorted(os.listdir(final_results_path)):
    if "behavior" in file.lower() or "fusion" in file.lower() or "text" in file.lower():
        print(file)

FINAL RESULTS FILES:

behavioral_common_test_results.csv
behavioral_final_results.csv
behavioral_lobo_test_results.csv
text_lobo_test_results.csv
text_lobo_test_results_with_ids.csv


In [29]:
y_lobo_val = behavior_lobo_val_results["label"].values

In [32]:
# ==========================================
# LOAD BOTH LOBO VALIDATION RESULTS
# ==========================================

behavior_lobo_val_results = pd.read_csv(
    os.path.join(
        final_results_path,
        "behavioral_lobo_val_results.csv"
    )
)

text_lobo_val_results = pd.read_csv(
    os.path.join(
        final_results_path,
        "text_lobo_val_results_with_ids.csv"
    )
)

print("Behavior validation:", behavior_lobo_val_results.shape)
print("Text validation:", text_lobo_val_results.shape)

Behavior validation: (1399, 3)
Text validation: (1399, 5)


In [33]:
# ==========================================
# EXTRACT ALIGNED VALIDATION ARRAYS
# ==========================================

behavior_lobo_val_prob = (
    behavior_lobo_val_results["behavior_probability"].values
)

text_lobo_val_prob = (
    text_lobo_val_results["text_probability"].values
)

y_lobo_val = (
    behavior_lobo_val_results["label"].values
)

print("Behavior probabilities:", len(behavior_lobo_val_prob))
print("Text probabilities:", len(text_lobo_val_prob))
print("Labels:", len(y_lobo_val))

Behavior probabilities: 1399
Text probabilities: 1399
Labels: 1399


In [34]:
print(
    "Labels identical:",
    np.array_equal(
        behavior_lobo_val_results["label"].values,
        text_lobo_val_results["label"].values
    )
)

Labels identical: True


In [35]:
# ==========================================
# 16. LOBO WEIGHTED FUSION — VALIDATION
# ==========================================

weights = np.arange(0.0, 1.01, 0.05)

weighted_lobo_val_results = []

for behavior_weight in weights:

    text_weight = 1 - behavior_weight

    fusion_prob = (
        behavior_weight * behavior_lobo_val_prob
        + text_weight * text_lobo_val_prob
    )

    fusion_pred = (
        fusion_prob >= 0.5
    ).astype(int)

    weighted_lobo_val_results.append({
        "behavior_weight": behavior_weight,
        "text_weight": text_weight,
        "accuracy": accuracy_score(
            y_lobo_val,
            fusion_pred
        ),
        "precision": precision_score(
            y_lobo_val,
            fusion_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_lobo_val,
            fusion_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_lobo_val,
            fusion_pred,
            zero_division=0
        )
    })

weighted_lobo_val_results = pd.DataFrame(
    weighted_lobo_val_results
)

print("==========================================")
print("LOBO WEIGHTED FUSION — VALIDATION")
print("==========================================")

display(
    weighted_lobo_val_results
    .sort_values("f1", ascending=False)
    .head(10)
)

LOBO WEIGHTED FUSION — VALIDATION


,behavior_weight,text_weight,accuracy,precision,recall,f1
10,0.50,0.50,0.999285,1.000000,0.999154,0.999577
9,0.45,0.55,0.997856,1.000000,0.997462,0.998729
8,0.40,0.60,0.995711,0.998305,0.996616,0.997460
7,0.35,0.65,0.994996,0.998304,0.995770,0.997035
6,0.30,0.70,0.994282,0.998302,0.994924,0.996610
11,0.55,0.45,0.994282,0.999150,0.994078,0.996607
4,0.20,0.80,0.993567,0.997455,0.994924,0.996188
5,0.25,0.75,0.993567,0.997455,0.994924,0.996188
12,0.60,0.40,0.992852,0.997453,0.994078,0.995763
13,0.65,0.35,0.992852,0.997453,0.994078,0.995763


In [36]:
# ==========================================
# 17. LOBO WEIGHTED FUSION — FINAL TEST
# ==========================================

# Frozen weight selected ONLY from validation
behavior_weight = 0.50
text_weight = 0.50

# Test probabilities
behavior_test_prob = lobo_fusion["behavior_probability"].values
text_test_prob = lobo_fusion["text_probability"].values

# Ground truth
y_lobo_test = lobo_fusion["label"].values

# Weighted fusion
weighted_lobo_test_prob = (
    behavior_weight * behavior_test_prob
    + text_weight * text_test_prob
)

weighted_lobo_test_pred = (
    weighted_lobo_test_prob >= 0.5
).astype(int)

# Metrics
weighted_lobo_accuracy = accuracy_score(
    y_lobo_test,
    weighted_lobo_test_pred
)

weighted_lobo_precision = precision_score(
    y_lobo_test,
    weighted_lobo_test_pred,
    zero_division=0
)

weighted_lobo_recall = recall_score(
    y_lobo_test,
    weighted_lobo_test_pred,
    zero_division=0
)

weighted_lobo_f1 = f1_score(
    y_lobo_test,
    weighted_lobo_test_pred,
    zero_division=0
)

weighted_lobo_cm = confusion_matrix(
    y_lobo_test,
    weighted_lobo_test_pred
)

print("==========================================")
print("FINAL LOBO WEIGHTED FUSION — 50:50")
print("==========================================")

print(f"Accuracy : {weighted_lobo_accuracy:.6f}")
print(f"Precision: {weighted_lobo_precision:.6f}")
print(f"Recall   : {weighted_lobo_recall:.6f}")
print(f"F1 Score : {weighted_lobo_f1:.6f}")

print("\nConfusion Matrix:")
print(weighted_lobo_cm)

print("\nClassification Report:")
print(
    classification_report(
        y_lobo_test,
        weighted_lobo_test_pred,
        target_names=["Genuine", "Bot"]
    )
)

FINAL LOBO WEIGHTED FUSION — 50:50
Accuracy : 0.970128
Precision: 0.991054
Recall   : 0.968770
F1 Score : 0.979785

Confusion Matrix:
[[1055   28]
 [ 100 3102]]

Classification Report:
              precision    recall  f1-score   support

     Genuine       0.91      0.97      0.94      1083
         Bot       0.99      0.97      0.98      3202

    accuracy                           0.97      4285
   macro avg       0.95      0.97      0.96      4285
weighted avg       0.97      0.97      0.97      4285



In [37]:
# ==========================================
# 18. BEHAVIOR MODEL — RELIABILITY ANALYSIS
# ==========================================

# Behavioral LOBO test data
behavior_prob = lobo_fusion["behavior_probability"].values
y_true = lobo_fusion["label"].values

# Confidence = distance from 0.5
behavior_confidence = np.abs(
    behavior_prob - 0.5
) * 2

# Frozen threshold selected from validation
confidence_threshold = 0.95

# Automatic = high confidence
automatic_mask = (
    behavior_confidence >= confidence_threshold
)

# Review = low confidence
review_mask = ~automatic_mask

# Predictions
behavior_pred = (
    behavior_prob >= 0.5
).astype(int)

# ------------------------------------------
# Automatic performance
# ------------------------------------------

automatic_accuracy = accuracy_score(
    y_true[automatic_mask],
    behavior_pred[automatic_mask]
)

automatic_count = automatic_mask.sum()
review_count = review_mask.sum()

automatic_coverage = (
    automatic_count / len(y_true)
) * 100

review_coverage = (
    review_count / len(y_true)
) * 100

# ------------------------------------------
# Review performance
# ------------------------------------------

review_accuracy = accuracy_score(
    y_true[review_mask],
    behavior_pred[review_mask]
)

print("==========================================")
print("BEHAVIOR MODEL — LOBO RELIABILITY")
print("==========================================")

print(f"Confidence threshold : {confidence_threshold}")

print("\nAutomatic:")
print(f"Count    : {automatic_count}")
print(f"Coverage : {automatic_coverage:.4f}%")
print(f"Accuracy : {automatic_accuracy:.4f}")

print("\nReview:")
print(f"Count    : {review_count}")
print(f"Coverage : {review_coverage:.4f}%")
print(f"Accuracy : {review_accuracy:.4f}")

BEHAVIOR MODEL — LOBO RELIABILITY
Confidence threshold : 0.95

Automatic:
Count    : 4088
Coverage : 95.4026%
Accuracy : 0.9963

Review:
Count    : 197
Coverage : 4.5974%
Accuracy : 0.7360


In [38]:
# ==========================================
# 19. ERROR CAPTURE BY RELIABILITY LAYER
# ==========================================

# Total model errors
total_errors = np.sum(
    behavior_pred != y_true
)

# Errors in automatic group
automatic_errors = np.sum(
    behavior_pred[automatic_mask] != y_true[automatic_mask]
)

# Errors in review group
review_errors = np.sum(
    behavior_pred[review_mask] != y_true[review_mask]
)

# Percentage of all errors sent to review
error_capture_rate = (
    review_errors / total_errors
) * 100

# Error rates within each group
automatic_error_rate = (
    automatic_errors / automatic_count
) * 100

review_error_rate = (
    review_errors / review_count
) * 100
     
print("==========================================")
print("RELIABILITY ERROR CAPTURE")
print("==========================================")

print(f"Total model errors       : {total_errors}")
print(f"Automatic errors         : {automatic_errors}")
print(f"Review errors            : {review_errors}")

print(f"\nError capture by review  : {error_capture_rate:.2f}%")

print(f"\nAutomatic error rate     : {automatic_error_rate:.4f}%")
print(f"Review error rate        : {review_error_rate:.4f}%")

RELIABILITY ERROR CAPTURE
Total model errors       : 67
Automatic errors         : 15
Review errors            : 52

Error capture by review  : 77.61%

Automatic error rate     : 0.3669%
Review error rate        : 26.3959%


In [39]:
# ==========================================
# 20. REVIEW CASE — BEHAVIOR VS TEXT
# ==========================================

review_data = lobo_fusion.loc[review_mask].copy()

review_behavior_pred = (
    review_data["behavior_probability"] >= 0.5
).astype(int)

review_text_pred = (
    review_data["text_probability"] >= 0.5
).astype(int)

review_true = review_data["label"].values

# ------------------------------------------
# Basic counts
# ------------------------------------------

behavior_correct = (
    review_behavior_pred.values == review_true
)

text_correct = (
    review_text_pred.values == review_true
)

print("==========================================")
print("REVIEW CASE — BEHAVIOR VS TEXT")
print("==========================================")

print("Review cases:", len(review_data))

print("\nBehavior model:")
print("Correct:", behavior_correct.sum())
print("Errors :", (~behavior_correct).sum())
print(
    "Accuracy:",
    accuracy_score(
        review_true,
        review_behavior_pred
    )
)

print("\nText model:")
print("Correct:", text_correct.sum())
print("Errors :", (~text_correct).sum())
print(
    "Accuracy:",
    accuracy_score(
        review_true,
        review_text_pred
    )
)

# ------------------------------------------
# Agreement
# ------------------------------------------

agreement = (
    review_behavior_pred.values ==
    review_text_pred.values
)

print("\nModel agreement:")
print("Agree    :", agreement.sum())
print("Disagree :", (~agreement).sum())

print(
    "Agreement rate:",
    agreement.mean()
)

# ------------------------------------------
# Text performance when models disagree
# ------------------------------------------

disagree_mask = ~agreement

if disagree_mask.sum() > 0:

    print("\nWhen models disagree:")

    print(
        "Cases:",
        disagree_mask.sum()
    )

    print(
        "Behavior accuracy:",
        accuracy_score(
            review_true[disagree_mask],
            review_behavior_pred.values[disagree_mask]
        )
    )

    print(
        "Text accuracy:",
        accuracy_score(
            review_true[disagree_mask],
            review_text_pred.values[disagree_mask]
        )
    )

REVIEW CASE — BEHAVIOR VS TEXT
Review cases: 197

Behavior model:
Correct: 145
Errors : 52
Accuracy: 0.7360406091370558

Text model:
Correct: 93
Errors : 104
Accuracy: 0.4720812182741117

Model agreement:
Agree    : 93
Disagree : 104
Agreement rate: 0.4720812182741117

When models disagree:
Cases: 104
Behavior accuracy: 0.75
Text accuracy: 0.25


In [40]:
# ==========================================
# 21. FINAL RELIABILITY-AWARE DECISION
# ==========================================

# Behavioral probability and prediction
behavior_prob = lobo_fusion["behavior_probability"].values
behavior_pred = (behavior_prob >= 0.5).astype(int)

# Confidence of behavioral prediction
behavior_confidence = np.abs(behavior_prob - 0.5) * 2

# Reliability threshold selected from validation
reliability_threshold = 0.95

# Automatic vs review
automatic_mask = behavior_confidence >= reliability_threshold
review_mask = ~automatic_mask

# Final decision
final_prediction = behavior_pred.copy()

# Decision label
decision = np.where(
    automatic_mask,
    "Automatic",
    "Review"
)

# Create final results table
final_decision_results = lobo_fusion[
    ["id", "label", "account_type",
     "behavior_probability",
     "text_probability"]
].copy()

final_decision_results["behavior_confidence"] = behavior_confidence
final_decision_results["behavior_prediction"] = behavior_pred
final_decision_results["decision"] = decision

print("==========================================")
print("FINAL RELIABILITY-AWARE DECISION")
print("==========================================")

print("\nTotal accounts:", len(final_decision_results))

print(
    "Automatic:",
    automatic_mask.sum(),
    f"({automatic_mask.mean()*100:.2f}%)"
)

print(
    "Review:",
    review_mask.sum(),
    f"({review_mask.mean()*100:.2f}%)"
)

# ------------------------------------------
# Accuracy of each decision group
# ------------------------------------------

y_true = final_decision_results["label"].values

automatic_accuracy = accuracy_score(
    y_true[automatic_mask],
    final_prediction[automatic_mask]
)

review_accuracy = accuracy_score(
    y_true[review_mask],
    final_prediction[review_mask]
)

overall_accuracy = accuracy_score(
    y_true,
    final_prediction
)

print("\nAutomatic accuracy:")
print(f"{automatic_accuracy*100:.2f}%")

print("\nReview accuracy:")
print(f"{review_accuracy*100:.2f}%")

print("\nOverall behavioral accuracy:")
print(f"{overall_accuracy*100:.2f}%")

FINAL RELIABILITY-AWARE DECISION

Total accounts: 4285
Automatic: 4088 (95.40%)
Review: 197 (4.60%)

Automatic accuracy:
99.63%

Review accuracy:
73.60%

Overall behavioral accuracy:
98.44%


In [41]:
# ==========================================
# 22A. CHECK EXPLAINABILITY PACKAGE
# ==========================================

try:
    import shap
    print("SHAP version:", shap.__version__)
    print("SHAP is ready.")
except ImportError:
    print("SHAP is not installed.")
    print("Run: pip install shap")

SHAP is not installed.
Run: pip install shap


In [42]:
# ==========================================
# 22. MASTER FUSION EXPERIMENT COMPARISON
# ==========================================

fusion_comparison = pd.DataFrame([
    {
        "Experiment": "Behavior-only",
        "Validation F1": np.nan,
        "Unseen Accuracy": 98.44,
        "Unseen F1": 98.95
    },
    {
        "Experiment": "Text-only",
        "Validation F1": np.nan,
        "Unseen Accuracy": 75.01,
        "Unseen F1": 82.06
    },
    {
        "Experiment": "50:50 Fusion",
        "Validation F1": 99.96,
        "Unseen Accuracy": 97.01,
        "Unseen F1": 97.98
    },
    {
        "Experiment": "Fine Weighted Fusion (48:52)",
        "Validation F1": 100.00,
        "Unseen Accuracy": 86.53,
        "Unseen F1": 90.55
    },
    {
        "Experiment": "Gated Fusion",
        "Validation F1": 99.96,
        "Unseen Accuracy": 95.61,
        "Unseen F1": 97.05
    },
    {
        "Experiment": "Confidence-aware Fusion",
        "Validation F1": 99.92,
        "Unseen Accuracy": 97.11,
        "Unseen F1": 98.04
    },
    {
        "Experiment": "Conservative Text Override",
        "Validation F1": 99.96,
        "Unseen Accuracy": 95.73,
        "Unseen F1": 97.13
    },
    {
        "Experiment": "Adaptive Fusion",
        "Validation F1": 99.66,
        "Unseen Accuracy": 79.58,
        "Unseen F1": 84.92
    }
])

print("==========================================")
print("MASTER FUSION EXPERIMENT COMPARISON")
print("==========================================")

display(
    fusion_comparison.sort_values(
        "Unseen Accuracy",
        ascending=False
    ).reset_index(drop=True)
)

MASTER FUSION EXPERIMENT COMPARISON


,Experiment,Validation F1,Unseen Accuracy,Unseen F1
0,Behavior-only,NaN,98.44,98.95
1,Confidence-aware Fusion,99.92,97.11,98.04
2,50:50 Fusion,99.96,97.01,97.98
3,Conservative Text Override,99.96,95.73,97.13
4,Gated Fusion,99.96,95.61,97.05
5,Fine Weighted Fusion (48:52),100.00,86.53,90.55
6,Adaptive Fusion,99.66,79.58,84.92
7,Text-only,NaN,75.01,82.06


In [43]:
# ==========================================
# FINAL CONCLUSION FROM FUSION EXPERIMENTS
# ==========================================

print("""
FINAL FUSION CONCLUSION
-----------------------

1. Fusion improves performance on the conventional common split.

2. On the unseen Fake Followers evaluation, behavioral-only
   detection generalizes better than the fusion strategies tested.

3. The behavioral model achieved 98.44% accuracy on unseen
   Fake Followers.

4. A reliability layer allows 95.40% of accounts to be processed
   automatically at 99.63% accuracy.

5. The review layer captures 77.61% of the behavioral model's errors.

6. Textual predictions were not reliable enough to automatically
   override behavioral predictions in uncertain cases.

7. Therefore, the final architecture uses behavioral analysis as
   the primary detector, confidence-based reliability assessment
   for decision routing, and textual analysis as supplementary
   evidence during review.
""")


FINAL FUSION CONCLUSION
-----------------------

1. Fusion improves performance on the conventional common split.

2. On the unseen Fake Followers evaluation, behavioral-only
   detection generalizes better than the fusion strategies tested.

3. The behavioral model achieved 98.44% accuracy on unseen
   Fake Followers.

4. A reliability layer allows 95.40% of accounts to be processed
   automatically at 99.63% accuracy.

5. The review layer captures 77.61% of the behavioral model's errors.

6. Textual predictions were not reliable enough to automatically
   override behavioral predictions in uncertain cases.

7. Therefore, the final architecture uses behavioral analysis as
   the primary detector, confidence-based reliability assessment
   for decision routing, and textual analysis as supplementary
   evidence during review.

